In [1]:
import os

In [2]:
%pwd

'/Users/abhikchoudhury/Library/CloudStorage/OneDrive-IBM/Badlo/Krish Naik Projects/AI-Powered Content Summarisation/Content-Summarizer_Abhik/Text_Summarizer_HF/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/Users/abhikchoudhury/Library/CloudStorage/OneDrive-IBM/Badlo/Krish Naik Projects/AI-Powered Content Summarisation/Content-Summarizer_Abhik/Text_Summarizer_HF'

In [ ]:
#This is basically a dataclass that will be used to define the configuration for data ingestion. It is an entity which we will later put in the entity file.
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path #root directory for data ingestion   return type is Path
    source_URL: str #source URL for data ingestion  return type is str
    local_data_file: Path #local data file for data ingestion  return type is Path
    unzip_dir: Path #unzip directory for data ingestion  return type is Path

#once this notebook is executed, we need to move this file to src/textSummarizer/entities/__init__.py

In [6]:
from textSummarizer.constants import *
from textSummarizer.utils.common import read_yaml, create_directories

In [ ]:
#In detail, the main purpose of this class is to manage the configuration and parameters for the data ingestion process. It reads the configuration and parameters from YAML files and 
# provides methods to retrieve specific configuration objects.
class ConfigurationManager:
    #This method is the constructor of the class. It takes two parameters: config_path and params_path, which are paths to the configuration and parameters YAML files, respectively.
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    #This method retrieves the data ingestion configuration from the configuration file and creates the necessary directories. and returns the data ingestion configuration.

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion

        create_directories([config.root_dir])

#the DataIngestionConfig is a dataclass that is defined in the constants file. It is used to store the configuration for the data ingestion process.
        #Why do we define it in the constants file? Because we want to make sure that the configuration is consistent across the project.
        data_ingestion_config = DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir 
        )

        return data_ingestion_config

#once this notebook is executed, we need to move this file to src/textSummarizer/config/configuration.py

In [8]:
import os
import urllib.request as request
import zipfile
from textSummarizer.logging import logger
from textSummarizer.utils.common import get_size

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config


    
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download! with following info: \n{headers}")
        else:
            logger.info(f"File already exists of size: {get_size(Path(self.config.local_data_file))}")  

        
    
    def extract_zip_file(self):
        """
        zip_file_path: str
        Extracts the zip file into the data directory
        Function returns None
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

#once this notebook is run, we will move the components to the components folder in Src(data_ingestion.py). This concludes the components creation for data ingestion.
#Now we will create the pipeline for data ingestion.

In [ ]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

#Once this notebook is run, the pipeline code will be moved to src/textSummarizer/pipeline/data_ingestion_pipeline.py

[2026-05-15 01:43:47,817: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-15 01:43:47,817: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-15 01:43:47,818: INFO: common: created directory at: artifacts]
[2026-05-15 01:43:47,818: INFO: common: created directory at: artifacts/data_ingestion]
[2026-05-15 01:43:47,818: INFO: 2756104762: File already exists of size:  ~ 7718 KB]
